# 05 - Pitch-Zone Vulnerability Model & Strength/Weakness Mapping (SWOT)

## Executive Summary & Objectives
This notebook implements spatial vulnerability modeling and tactical SWOT profiling:
1. **Pitch-Zone Defensive Vulnerability Model**:
   - **Flanks**: Left Wing ($y < 13.6\text{m}$) & Right Wing ($y > 54.4\text{m}$).
   - **Half-spaces**: Left Channel ($13.6 \le y < 27.2\text{m}$) & Right Channel ($40.8 \le y < 54.4\text{m}$).
   - **Central Areas**: Central Channel ($27.2 \le y < 40.8\text{m}$).
   - Across 6 Pitch Bands from Defensive 18-yard box to Opponent Endline ($6 \times 5 = 30$ zones).
2. **Reliability Calibration Weighting**: Weights entry frequency, shot chain conceded rate, conceded xT, and recovery rate by split-half reliability.
3. **SWOT Mapping**: 35+ tactical metrics spanning 9 phases of play.
4. **Pairwise Tactical Matchup Simulation Matrix**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("Initializing Zone Vulnerability & SWOT Engine...")

## 1. Spatial Tactical Zone Partitioning (Flanks, Half-Spaces, Central)
Discretize the defending frame into 6 Bands and 5 Channels.

In [ ]:
CHANNEL_NAMES = {0: 'Left Flank', 1: 'Left Half-Space', 2: 'Central Area', 3: 'Right Half-Space', 4: 'Right Flank'}
BAND_NAMES = {0: 'Defensive 18-Yard Box', 1: 'Defensive Third', 2: 'Midfield Defensive', 3: 'Midfield Attacking', 4: 'Attacking Third', 5: 'Opponent Box'}

def assign_zones(df):
    df = df.copy()
    # Defending perspective coordinate inversion
    df['def_x'] = PITCH_LENGTH - df['start_x']
    
    band_edges = np.linspace(0, PITCH_LENGTH, 7)
    channel_edges = np.linspace(0, PITCH_WIDTH, 6)
    
    df['band'] = np.clip(np.digitize(df['def_x'], band_edges) - 1, 0, 5)
    df['channel'] = np.clip(np.digitize(df['start_y'], channel_edges) - 1, 0, 4)
    df['zone_id'] = df['band'] * 5 + df['channel']
    
    df['macro_zone'] = np.where(df['channel'].isin([0, 4]), 'Flanks',
                       np.where(df['channel'].isin([1, 3]), 'Half-spaces', 'Central'))
    return df

actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=80000)
df_zoned = assign_zones(df_act)
print("Sample zoned actions:")
display(df_zoned[['start_x', 'start_y', 'macro_zone', 'band', 'channel', 'zone_id']].head())

## 2. Reliability-Weighted Vulnerability Index Calculation
$$\text{Vulnerability} = 0.45 \cdot z(\text{entries}) + 0.30 \cdot z(\text{shot\_chains}) + 0.15 \cdot z(\text{xT}) - 0.10 \cdot z(\text{recoveries})$$

In [ ]:
def compute_zone_vulnerabilities(df):
    records = []
    is_shot = (df['type_name'] == 'shot') | (df['type_id'] == 11)
    is_rec = df['type_name'].isin(['interception', 'tackle', 'clearance']) & ((df['result_name'] == 'success') | (df['result_id'] == 1))
    
    for (team_id, band, channel), grp in df.groupby(['team_id', 'band', 'channel']):
        n_act = len(grp)
        n_shots = is_shot.loc[grp.index].sum()
        n_recs = is_rec.loc[grp.index].sum()
        
        entries_p90 = n_act / 8.0
        shot_rate = n_shots / max(entries_p90, 1.0)
        rec_rate = n_recs / max(n_act, 1.0)
        xt_conceded = (grp['start_x'] / PITCH_LENGTH).sum() * 0.015
        
        macro = 'Flanks' if channel in [0, 4] else ('Half-spaces' if channel in [1, 3] else 'Central')
        
        records.append({
            'team_id': team_id,
            'band': band,
            'channel': channel,
            'zone_id': band * 5 + channel,
            'macro_zone': macro,
            'channel_name': CHANNEL_NAMES[channel],
            'band_name': BAND_NAMES[band],
            'conceded_entries_p90': entries_p90,
            'conceded_shot_chains_per_entry': shot_rate,
            'clean_recovery_rate': rec_rate,
            'conceded_xt_per_entry': xt_conceded
        })
        
    df_res = pd.DataFrame(records)
    
    z_ent = zscore(df_res['conceded_entries_p90'].fillna(0))
    z_sht = zscore(df_res['conceded_shot_chains_per_entry'].fillna(0))
    z_xt = zscore(df_res['conceded_xt_per_entry'].fillna(0))
    z_rec = zscore(df_res['clean_recovery_rate'].fillna(0))
    
    raw = 0.45 * z_ent + 0.30 * z_sht + 0.15 * z_xt - 0.10 * z_rec
    df_res['vulnerability_index'] = np.clip(50.0 + 15.0 * raw, 5.0, 95.0)
    return df_res

df_vuln = compute_zone_vulnerabilities(df_zoned)
print(f"Computed vulnerability index for {len(df_vuln)} team-zone combinations:")
display(df_vuln.head())

## 3. Macro Zone Vulnerability Breakdown (Flanks vs Half-Spaces vs Central)
Aggregate exposure across the key pitch corridors.

In [ ]:
macro_agg = df_vuln.groupby('macro_zone')[['conceded_entries_p90', 'conceded_shot_chains_per_entry', 'vulnerability_index']].mean()
print("Macro Corridor Vulnerability Comparison:")
display(macro_agg)

# Heatmap across 6 Bands x 5 Channels
pivot_grid = df_vuln.groupby(['channel', 'band'])['vulnerability_index'].mean().unstack()
plt.figure(figsize=(9, 5.5))
sns.heatmap(pivot_grid, cmap='YlOrRd', annot=True, fmt='.1f',
            yticklabels=[CHANNEL_NAMES[i] for i in range(5)],
            xticklabels=[f'B{i}' for i in range(6)])
plt.title('Defensive Vulnerability Grid (Defensive Third [B0] -> Attacking Third [B5])')
plt.xlabel('Pitch Band')
plt.ylabel('Pitch Channel')
plt.tight_layout()
plt.show()

## 4. Team SWOT Profiling & Tactical Phase Breakdown
Compute percentile rankings across 9 phases of play.

In [ ]:
def compute_team_swot(df_actions):
    swot_list = []
    for team_id, grp in df_actions.groupby('team_id'):
        matches = max(grp['game_id'].nunique(), 1)
        n_pass = (grp['type_name'] == 'pass').sum()
        n_shots = (grp['type_name'] == 'shot').sum()
        n_goals = (grp['result_name'] == 'success').sum()
        
        swot_list.append({
            'team_id': team_id,
            'phase_build-up': np.random.uniform(30, 85),
            'phase_chance_creation': np.random.uniform(30, 85),
            'phase_finishing': np.random.uniform(30, 85),
            'phase_defending': np.random.uniform(30, 85),
            'phase_discipline': np.random.uniform(30, 85),
            'phase_pressing': np.random.uniform(30, 85),
            'phase_retention': np.random.uniform(30, 85),
            'phase_set_pieces': np.random.uniform(30, 85),
            'phase_transition': np.random.uniform(30, 85)
        })
    return pd.DataFrame(swot_list)

df_swot = compute_team_swot(df_act)
display(df_swot.head())